## Apex Wealth Data Pipeline

In [ ]:
!pip install request
!pip install python-dotenv

In [ ]:
import requests
import pandas as pd 
from dotenv import load_dotenv
import os
import psycopg2
from sqlalchemy import create_engine

## Extraction

In [ ]:
load_dotenv
api_key = os.getenv('API_KEY')

api_key

In [ ]:
# define url endpoint
symbol = 'AAPL'
url = f'https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=4'

In [ ]:
# make a request to the API 
response = requests.get(url)
response.status_code

In [ ]:
# get data in json format
data = response.json()
data

In [ ]:
# tranfer data to a dataframe
def transform_data(data):
    
    
    # extract values from json
    time_series = data['values']
    
    #convert to dataframe
    df = pd.DataFrame(time_series)
    
    # convert columns to appropriate data type
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    df = df.astype({
        'open':'float',
        'high':'float',
        'low':'float',
        'close':'float',
        'volume':'int'
    })
    return df

In [ ]:
df = transform_data(data)
df

### Extract from multiple symbols

In [ ]:
symbols =['AAPL','MSFT','GOOGL','AMZN']
all_data = []

def fetch_data(symbol,api_key):
    url = f'https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=4'
    response = requests.get(url)
    response.raise_for_status() # raise an error for bad responses
    data = response.json()
    
    #check if the api response is ok
    if data['status'] != 'ok':
        print(f"error with {symbol}: {data['message']}")
    return data
    

In [ ]:
symbols =['AAPL','MSFT','GOOGL','AMZN']

for symbol in symbols:
    data = fetch_data(symbol,api_key)
    df = transform_data(data)
    df['sym'] = symbol # add a column for the symbol
    all_data.append(df)

In [ ]:
all_data

In [ ]:
# using concat - joining or stacking all the data together
all_df = pd.concat(all_data,ignore_index=True)

In [ ]:
all_df

### Loading

In [ ]:
# database credentials
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_PORT = os.getenv('DB_PORT')

In [ ]:
DB_NAME

In [ ]:
#create connection string with url
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

# load data to data base 
all_df.to_sql('stockprices_data',engine,if_exists='append',index=False)

print('data loaded successfully')

In [ ]:
# save to csv
all_df.to_csv('stockprice_data.csv',index=False)